# Picotron Kaggle GPU smoke test

Run this notebook with Kaggle GPU enabled (expected hardware: 2× T4). It verifies real CUDA behavior for hardware selection, single-GPU pretraining, spawned-process DDP, checkpoint resume, and script-first SFT.

The notebook uses deterministic synthetic batches so a falling loss is an unambiguous training signal; Hugging Face streaming is intentionally not part of this GPU-core smoke test.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import traceback
from pathlib import Path

def find_repo() -> Path:
    candidates = [
        Path(os.environ.get('PICOTRON_REPO', '')),
        Path('/kaggle/working/picotron'),
        Path('/kaggle/input/picotron'),
        Path.cwd(),
    ]
    for candidate in candidates:
        if candidate and (candidate / 'requirements.txt').is_file():
            return candidate
    raise FileNotFoundError('Set PICOTRON_REPO to the uploaded/cloned Picotron directory.')

REPO = find_repo()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
sys.path.insert(0, str(REPO / 'src'))
WORKDIR = Path('/kaggle/working/picotron_smoke')
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULTS: dict[str, tuple[bool, str]] = {}

def run_section(name, callback):
    try:
        callback()
        RESULTS[name] = (True, 'PASS')
        print(f'✅ {name}: PASS')
    except Exception as error:
        RESULTS[name] = (False, f'{type(error).__name__}: {error}')
        print(f'❌ {name}: {error}')
        traceback.print_exc()


## 1. Hardware policy

Pass: both GPUs report compute capability `(7, 5)`, the selected training dtype is `float16`, never `bfloat16`, and the attention/Triton reports are printed.

In [ ]:
import torch

from picotron.utils.hardware import (
    detect_attention_backend,
    detect_triton_support,
    get_gpu_compute_capability,
    select_training_dtype,
)

def verify_hardware():
    assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
    assert torch.cuda.device_count() >= 2, 'This notebook expects two T4 GPUs.'
    for device in range(2):
        capability = get_gpu_compute_capability(device)
        dtype = select_training_dtype(device)
        print(f'GPU {device}: {torch.cuda.get_device_name(device)}, capability={capability}, dtype={dtype}')
        assert capability == (7, 5), f'Expected Turing T4 (7.5), got {capability}'
        assert dtype is torch.float16
        assert dtype is not torch.bfloat16
    print('attention:', detect_attention_backend(0))
    print('triton (explicitly disabled):', detect_triton_support(enabled=False, device=0))

run_section('hardware detection', verify_hardware)


## 2. Single-GPU pretraining and checkpoint resume

Pass: a tiny decoder learns repeated synthetic sequences on `cuda:0`; a checkpoint restores exact weights into a fresh model; resumed loss continues to decrease.

In [ ]:
from dataclasses import replace

from picotron.config.config import PicotronConfig
from picotron.models.picotron_decoder import PicotronDecoderModel
from picotron.serialize.checkpoint import load_checkpoint
from picotron.training.train_loop import train

SMOKE_CONFIG = PicotronConfig(
    checkpoints={'checkpoint_interval': 10},
    model={'model_config': {
        'vocab_size': 64, 'hidden_size': 32, 'intermediate_size': 64,
        'num_hidden_layers': 2, 'num_attention_heads': 4,
    }},
    optimizer={'learning_rate_scheduler': {'learning_rate': 3e-3}},
    parallelism={'dp': 1, 'zero_stage': 0},
    tokens={'sequence_length': 16, 'micro_batch_size': 2, 'train_steps': 30},
    data={'vocab_size': 64},
    logging={},
    general={'run': 'kaggle_smoke'},
)

def repeated_batches(config, count, offset=0):
    model_config = config.model.model_config
    tokens = (torch.arange(config.tokens.sequence_length) + offset).remainder(model_config.vocab_size)
    return [tokens.unsqueeze(0).repeat(config.tokens.micro_batch_size, 1) for _ in range(count)]

SINGLE_CHECKPOINT = WORKDIR / 'single_gpu.safetensors'

def verify_single_gpu_and_checkpoint():
    torch.manual_seed(123)
    model = PicotronDecoderModel(SMOKE_CONFIG)
    losses = train(model, repeated_batches(SMOKE_CONFIG, 30), SMOKE_CONFIG, device='cuda:0', max_steps=30)
    first, last = sum(losses[:5]) / 5, sum(losses[-5:]) / 5
    print(f'single-GPU first_5={first:.6f}, last_5={last:.6f}')
    assert last < first

    checkpoint_config = replace(
        SMOKE_CONFIG,
        checkpoints=replace(SMOKE_CONFIG.checkpoints, checkpoint_interval=10),
    )
    checkpoint_model = PicotronDecoderModel(checkpoint_config)
    checkpoint_losses = train(
        checkpoint_model, repeated_batches(checkpoint_config, 10), checkpoint_config,
        device='cuda:0', max_steps=10, checkpoint_path=str(SINGLE_CHECKPOINT),
    )
    saved_state = {name: value.detach().cpu().clone() for name, value in checkpoint_model.state_dict().items()}
    fresh_model = PicotronDecoderModel(checkpoint_config).to('cuda:0')
    fresh_optimizer = torch.optim.AdamW(
        fresh_model.parameters(),
        lr=checkpoint_config.optimizer.learning_rate_scheduler.learning_rate,
    )
    resumed_step = load_checkpoint(fresh_model, fresh_optimizer, SINGLE_CHECKPOINT)
    assert resumed_step == 10
    for name, value in saved_state.items():
        torch.testing.assert_close(value, fresh_model.state_dict()[name].detach().cpu(), rtol=0, atol=0)
    resumed_losses = train(
        fresh_model, repeated_batches(checkpoint_config, 10), checkpoint_config,
        optimizer=fresh_optimizer, device='cuda:0', max_steps=10,
    )
    print(f'checkpoint last={checkpoint_losses[-1]:.6f}, resumed first={resumed_losses[0]:.6f}, resumed last={resumed_losses[-1]:.6f}')
    assert resumed_losses[-1] < resumed_losses[0]

run_section('single-GPU pretraining + checkpoint resume', verify_single_gpu_and_checkpoint)


## 3. Two-GPU DDP

Kaggle notebooks cannot reliably launch notebook cells as spawn targets. This cell writes a small worker module and starts it as a subprocess; the worker uses `torch.multiprocessing.spawn`, NCCL, and Picotron's normal training loop. Pass: losses decrease and both ranks finish with identical parameters.

In [ ]:
DDP_WORKER = WORKDIR / 'ddp_worker.py'
DDP_RESULT = WORKDIR / 'ddp_result.pt'
DDP_WORKER.write_text(r'''
import os
import socket
import sys
from dataclasses import replace
from pathlib import Path

import torch
import torch.distributed as dist
import torch.multiprocessing as mp

REPO = Path(sys.argv[1])
OUTPUT = Path(sys.argv[2])
sys.path.insert(0, str(REPO / 'src'))
from picotron.config.config import PicotronConfig
from picotron.models.picotron_decoder import PicotronDecoderModel
from picotron.parallel.ddp import cleanup_distributed
from picotron.training.train_loop import train

def free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        return sock.getsockname()[1]

def worker(rank, world_size, port):
    os.environ.update({'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': str(port), 'RANK': str(rank), 'WORLD_SIZE': str(world_size), 'LOCAL_RANK': str(rank)})
    torch.cuda.set_device(rank)
    config = PicotronConfig(
        checkpoints={'checkpoint_interval': 100},
        model={'model_config': {
            'vocab_size': 64, 'hidden_size': 32, 'intermediate_size': 64,
            'num_hidden_layers': 2, 'num_attention_heads': 4,
        }},
        optimizer={'learning_rate_scheduler': {'learning_rate': 3e-3}},
        parallelism={'dp': 2, 'zero_stage': 0},
        tokens={'sequence_length': 16, 'micro_batch_size': 2, 'train_steps': 20},
        data={'vocab_size': 64}, logging={}, general={'run': 'kaggle_smoke_ddp'},
    )
    torch.manual_seed(321)
    model = PicotronDecoderModel(config)
    tokens = (torch.arange(config.tokens.sequence_length) + rank).remainder(config.model.model_config.vocab_size)
    batches = [tokens.unsqueeze(0).repeat(config.tokens.micro_batch_size, 1) for _ in range(20)]
    losses = train(model, batches, config, device=f'cuda:{rank}', max_steps=20)
    flat = torch.cat([parameter.detach().reshape(-1) for parameter in model.parameters()])
    gathered = [torch.empty_like(flat) for _ in range(world_size)]
    dist.all_gather(gathered, flat)
    if rank == 0:
        assert torch.allclose(gathered[0], gathered[1], rtol=1e-6, atol=1e-6)
        torch.save({'first_5': sum(losses[:5]) / 5, 'last_5': sum(losses[-5:]) / 5}, OUTPUT)
    dist.barrier()
    cleanup_distributed()

if __name__ == '__main__':
    mp.spawn(worker, args=(2, free_port()), nprocs=2, join=True)
''', encoding='utf-8')

def verify_ddp():
    subprocess.run([sys.executable, str(DDP_WORKER), str(REPO), str(DDP_RESULT)], check=True)
    result = torch.load(DDP_RESULT, map_location='cpu')
    print(f"DDP first_5={result['first_5']:.6f}, last_5={result['last_5']:.6f}")
    assert result['last_5'] < result['first_5']

run_section('two-GPU DDP', verify_ddp)


## 4. Script-first SFT

Pass: `run_sft()` loads the real CUDA checkpoint into a fresh model and reduces loss on a different supervised synthetic sequence. This is intentionally a direct Python API call, not YAML-driven.

In [ ]:
from torch.utils.data import Dataset

from picotron_sft import run_sft

class SupervisedSequenceDataset(Dataset):
    def __init__(self, config, count):
        self.config = config
        self.count = count
        self.sequence = torch.arange(config.tokens.sequence_length - 1, -1, -1).remainder(config.model.model_config.vocab_size)

    def __len__(self):
        return self.count

    def __getitem__(self, index):
        return {'input_ids': self.sequence.clone(), 'labels': self.sequence.clone()}

def verify_sft():
    model = PicotronDecoderModel(SMOKE_CONFIG)
    dataset = SupervisedSequenceDataset(SMOKE_CONFIG, count=40)
    losses = run_sft(
        model, dataset, base_checkpoint_path=str(SINGLE_CHECKPOINT),
        learning_rate=SMOKE_CONFIG.optimizer.learning_rate_scheduler.learning_rate,
        batch_size=SMOKE_CONFIG.tokens.micro_batch_size,
        num_steps=20, device='cuda:0',
    )
    first, last = sum(losses[:5]) / 5, sum(losses[-5:]) / 5
    print(f'SFT first_5={first:.6f}, last_5={last:.6f}')
    assert len(losses) == 20
    assert last < first

run_section('script-first SFT', verify_sft)


## Final result

A successful Kaggle run should show five green sections below. Any failure includes its exception message, which is useful evidence for the next debugging prompt.

In [ ]:
print('\nPICOTRON KAGGLE SMOKE TEST SUMMARY')
print('=' * 40)
for name, (passed, detail) in RESULTS.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'{status:4} | {name} | {detail}')
all_passed = bool(RESULTS) and all(passed for passed, _ in RESULTS.values())
print('=' * 40)
print('OVERALL:', 'PASS' if all_passed else 'FAIL')
assert all_passed, 'One or more GPU smoke-test sections failed; inspect the cells above.'
